# NHANES 2021-2023: Sleep and Blood Pressure
## Step 4: SQL Analysis with DuckDB
Replicating and extending key exploratory queries from Python EDA using SQL. DuckDB will allow direct querying of CSV files without requiring a traditional database setup.

In [7]:
import duckdb
import pandas as pd

# Connect to DuckDB and load the CSV directly
con = duckdb.connect() # Creating an in-memory database living on RAM
con.execute("CREATE TABLE nhanes AS SELECT * FROM '../Data/nhanes_clean.csv'") # SQL command to con

# Confirm the db loaded
result = con.execute("SELECT COUNT(*) FROM nhanes").fetchdf()
print(result)

   count_star()
0          5999


### Query 1: Distribution of Sleep Hours
Replicates EDA histogram. Shows most participants report 7-8 hours of sleep. 8.0 hours is the modal category (20.47% of sample).

In [8]:
result = con.execute("""
    SELECT 
        sleep_hours,
        COUNT(*) AS n_participants,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_sample
    FROM nhanes
    GROUP BY sleep_hours
    ORDER BY sleep_hours
""").fetchdf()

print(result)

    sleep_hours  n_participants  pct_of_sample
0           2.0              25           0.42
1           3.0              31           0.52
2           3.5              23           0.38
3           4.0              86           1.43
4           4.5              38           0.63
5           5.0             203           3.38
6           5.5             122           2.03
7           6.0             408           6.80
8           6.5             316           5.27
9           7.0             893          14.89
10          7.5             676          11.27
11          8.0            1228          20.47
12          8.5             564           9.40
13          9.0             691          11.52
14          9.5             178           2.97
15         10.0             289           4.82
16         10.5              47           0.78
17         11.0              91           1.52
18         11.5              11           0.18
19         12.0              34           0.57
20         12

### Query 2: Average BP by Sleep Group
Replicates EDA Boxplot. Average systolic BP is relatively stable across sleep  groups (~121-124 mmHg), consistent with the non-significant sleep coefficient in regression. Note: extreme sleep groups (<=3hrs, >=11.5hrs) have very small sample sizes (n<35) and should be interpreted cautiously.

In [9]:
result2 = con.execute("""
    SELECT
        sleep_hours,
        ROUND(AVG(systolic_bp), 2) AS avg_systolic_bp,
        ROUND(AVG(diastolic_bp), 2) AS avg_diastolic_bp,
        COUNT(*) AS n_participants
    FROM nhanes
    GROUP BY sleep_hours
    ORDER BY sleep_hours
""").fetchdf()

print(result2)

    sleep_hours  avg_systolic_bp  avg_diastolic_bp  n_participants
0           2.0           122.80             77.04              25
1           3.0           128.19             79.00              31
2           3.5           121.26             78.61              23
3           4.0           123.51             76.05              86
4           4.5           123.84             77.68              38
5           5.0           122.99             75.79             203
6           5.5           125.88             77.33             122
7           6.0           122.00             75.00             408
8           6.5           123.72             77.61             316
9           7.0           121.85             74.78             893
10          7.5           122.13             75.06             676
11          8.0           122.33             74.71            1228
12          8.5           120.10             73.46             564
13          9.0           123.37             74.82            

### Query 3: Hypertension Rates by Age Group
Hypertension diagnosis rates increase dramatically with age — from 5.95% in adults 18-30 to 55.84% in adults 61-80. The sharpest increase occurs between the 31-45 and 46-60 groups (16.83% → 40.62%), consistent with age-related arterial stiffness and the dominant age effect found in regression modeling.

In [10]:
result3 = con.execute("""
    SELECT
        CASE
            WHEN age BETWEEN 18 AND 30 THEN '18-30'
            WHEN age BETWEEN 31 AND 45 THEN '31-45'
            WHEN age BETWEEN 46 AND 60 THEN '46-60'
            WHEN age BETWEEN 61 AND 80 THEN '61-80'
        END AS age_group,
        COUNT(*) as n_participants,
        ROUND(AVG(systolic_bp), 2) AS avg_systolic_bp,
        ROUND(SUM(CASE WHEN hypertension_diagnosis = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_hypertensive
    FROM nhanes
    GROUP BY age_group
    ORDER BY age_group
""").fetchdf()

print(result3)

  age_group  n_participants  avg_systolic_bp  pct_hypertensive
0     18-30             958           112.09              5.95
1     31-45            1260           115.33             16.83
2     46-60            1317           123.61             40.62
3     61-80            2464           129.68             55.84


### Query 4: Average BP by Sex (Summary Query)
Males have higher average systolic BP (125.14 vs 120.36 mmHg) and higher hypertension rates (37.59% vs 35.31%) than females, consistent with regression 
findings (β=-4.82, OR=0.827). Average sleep hours are nearly identical between sexes (7.64 vs 7.79), confirming the non-significant sleep × sex main effect.

In [13]:
result4 = con.execute("""
    SELECT
        CASE WHEN sex = 1 THEN 'Male' ELSE 'Female' END AS sex_label,
        COUNT(*) as n_participants,
        ROUND(AVG(systolic_bp), 2) AS avg_systolic_bp, 
        ROUND(AVG(diastolic_bp), 2) AS avg_diastolic_bp,
        ROUND(AVG(sleep_hours), 2) AS avg_sleep_hours,
        ROUND(SUM(CASE WHEN hypertension_diagnosis = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_hypertensive
    FROM nhanes
    GROUP BY sex
    ORDER BY sex
""").fetchdf()

print(result4)

  sex_label  n_participants  avg_systolic_bp  avg_diastolic_bp  \
0      Male            2711           125.14             75.60   
1    Female            3288           120.36             74.53   

   avg_sleep_hours  pct_hypertensive  
0             7.64             37.59  
1             7.79             35.31  


### Query 5: Summary Statistics by Hypertension Diagnosis
Hypertensive participants are on average older (62.29 vs 46.56 years) and have higher BMI (31.68 vs 28.37) than non-hypertensive participants, consistent with  regression findings. Sleep hours are virtually identical between groups (7.71 vs 7.72), providing a clean summary of the null sleep finding.

In [14]:
result5 = con.execute("""
    SELECT
        CASE WHEN hypertension_diagnosis = 1 THEN 'Hypertensive' ELSE 'Not Hypertensive' END AS hypertension_label,
        COUNT(*) as n_participants,
        ROUND(AVG(systolic_bp), 2) AS avg_systolic_bp, 
        ROUND(AVG(diastolic_bp), 2) AS avg_diastolic_bp,
        ROUND(AVG(sleep_hours), 2) AS avg_sleep_hours,
        ROUND(AVG(bmi), 2) AS avg_bmi,
        ROUND(AVG(age), 2) AS avg_age
    FROM nhanes
    GROUP BY hypertension_diagnosis
    ORDER BY hypertension_diagnosis
""").fetchdf()
print(result5)

  hypertension_label  n_participants  avg_systolic_bp  avg_diastolic_bp  \
0       Hypertensive            2180           130.82             77.53   
1   Not Hypertensive            3819           117.79             73.58   

   avg_sleep_hours  avg_bmi  avg_age  
0             7.71    31.68    62.29  
1             7.72    28.37    46.56  


### Query 6: BP by Sleep Category and Sex
Replicates the significant sleep × sex interaction (p=0.025) found in modeling. Males show a slight protective/downward trend in systolic BP with more sleep (125.89 → 124.76 mmHg), while females show no consistent pattern (120.63 → 119.78 → 121.09 mmHg).Average age is nearly identical across all six groups (~52 years), ruling out age as a confound in this pattern.

In [15]:
result6 = con.execute("""
    SELECT
        CASE
            WHEN sleep_hours <= 6 THEN 'short'
            WHEN sleep_hours <= 8 THEN 'normal'
            ELSE 'long'
        END as sleep_group,
        CASE WHEN sex = 1 THEN 'Male' ELSE 'Female' END AS sex_label,
        COUNT(*) as n_participants,
        ROUND(AVG(systolic_bp), 2) AS avg_systolic_bp, 
        ROUND(AVG(diastolic_bp), 2) AS avg_diastolic_bp,
        ROUND(AVG(sleep_hours), 2) AS avg_sleep_hours,
        ROUND(AVG(bmi), 2) AS avg_bmi,
        ROUND(AVG(age), 2) AS avg_age
    FROM nhanes
    GROUP BY sleep_group, sex_label
    ORDER BY sex_label, sleep_group
""").fetchdf()
print(result6)


  sleep_group sex_label  n_participants  avg_systolic_bp  avg_diastolic_bp  \
0        long    Female            1147           121.09             74.18   
1      normal    Female            1652           119.78             74.36   
2       short    Female             489           120.63             75.96   
3        long      Male             803           124.76             74.80   
4      normal      Male            1461           125.13             75.94   
5       short      Male             447           125.89             75.96   

   avg_sleep_hours  avg_bmi  avg_age  
0             9.34    29.91    52.04  
1             7.48    29.61    52.44  
2             5.19    31.48    52.01  
3             9.36    28.50    53.01  
4             7.43    29.21    51.94  
5             5.22    29.62    52.37  


### Exporting Queries as SQL Files

In [17]:
import os

# Create sql folder
os.makedirs('../sql', exist_ok=True)
print("sql/ folder created")

sql/ folder created


In [18]:
queries = {
    "01_sleep_distribution.sql": """
-- Query 1: Sleep Hours Distribution
-- Shows the distribution of self-reported sleep hours across the sample.
-- Most participants report 7-8 hours; 8.0 hours is the modal category (20.47%).

SELECT 
    sleep_hours,
    COUNT(*) AS n_participants,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_sample
FROM nhanes
GROUP BY sleep_hours
ORDER BY sleep_hours;
""",

    "02_avg_bp_by_sleep.sql": """
-- Query 2: Average Blood Pressure by Sleep Group
-- Average systolic BP is stable across sleep groups (~121-124 mmHg),
-- consistent with the non-significant sleep coefficient in regression (p=0.499).
-- Note: extreme sleep groups (<=3hrs, >=11.5hrs) have very small sample sizes.

SELECT
    sleep_hours,
    ROUND(AVG(systolic_bp), 2) AS avg_systolic_bp,
    ROUND(AVG(diastolic_bp), 2) AS avg_diastolic_bp,
    COUNT(*) AS n_participants
FROM nhanes
GROUP BY sleep_hours
ORDER BY sleep_hours;
""",

    "03_hypertension_by_age.sql": """
-- Query 3: Hypertension Rates by Age Group
-- Hypertension rates increase dramatically with age, from 5.95% (18-30)
-- to 55.84% (61-80). Sharpest increase between 31-45 and 46-60 age groups,
-- consistent with age-related arterial stiffness and dominant age effect in regression.

SELECT
    CASE
        WHEN age BETWEEN 18 AND 30 THEN '18-30'
        WHEN age BETWEEN 31 AND 45 THEN '31-45'
        WHEN age BETWEEN 46 AND 60 THEN '46-60'
        WHEN age BETWEEN 61 AND 80 THEN '61-80'
    END AS age_group,
    COUNT(*) AS n_participants,
    ROUND(AVG(systolic_bp), 2) AS avg_systolic_bp,
    ROUND(SUM(CASE WHEN hypertension_diagnosis = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_hypertensive
FROM nhanes
GROUP BY age_group
ORDER BY age_group;
""",

    "04_summary_by_sex.sql": """
-- Query 4: Summary Statistics by Sex
-- Males have higher average systolic BP (125.14 vs 120.36 mmHg) and higher
-- hypertension rates (37.59% vs 35.31%). Sleep hours nearly identical between
-- sexes (7.64 vs 7.79), confirming non-significant sleep x sex main effect.

SELECT
    CASE WHEN sex = 1 THEN 'Male' ELSE 'Female' END AS sex_label,
    COUNT(*) AS n_participants,
    ROUND(AVG(systolic_bp), 2) AS avg_systolic_bp,
    ROUND(AVG(diastolic_bp), 2) AS avg_diastolic_bp,
    ROUND(AVG(sleep_hours), 2) AS avg_sleep_hours,
    ROUND(SUM(CASE WHEN hypertension_diagnosis = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_hypertensive
FROM nhanes
GROUP BY sex
ORDER BY sex;
""",

    "05_summary_by_hypertension.sql": """
-- Query 5: Summary Statistics by Hypertension Status
-- Hypertensive participants are older (62.29 vs 46.56 years) and have higher
-- BMI (31.68 vs 28.37). Sleep hours virtually identical (7.71 vs 7.72),
-- providing a clean single-number summary of the null sleep finding.

SELECT
    CASE WHEN hypertension_diagnosis = 1 THEN 'Hypertensive' ELSE 'Not Hypertensive' END AS diagnosis,
    COUNT(*) AS n_participants,
    ROUND(AVG(age), 2) AS avg_age,
    ROUND(AVG(bmi), 2) AS avg_bmi,
    ROUND(AVG(sleep_hours), 2) AS avg_sleep_hours,
    ROUND(AVG(systolic_bp), 2) AS avg_systolic_bp
FROM nhanes
GROUP BY hypertension_diagnosis
ORDER BY hypertension_diagnosis;
""",

    "06_bp_by_sleep_and_sex.sql": """
-- Query 6: Average BP by Sleep Category and Sex
-- Replicates sleep x sex interaction (p=0.025) from regression modeling.
-- Males show slight downward BP trend with more sleep (125.89 -> 124.76 mmHg).
-- Females show no consistent pattern. Age nearly identical across groups (~52 years).

SELECT
    CASE
        WHEN sleep_hours <= 6 THEN 'short'
        WHEN sleep_hours <= 8 THEN 'normal'
        ELSE 'long'
    END AS sleep_group,
    CASE WHEN sex = 1 THEN 'Male' ELSE 'Female' END AS sex_label,
    COUNT(*) AS n_participants,
    ROUND(AVG(systolic_bp), 2) AS avg_systolic_bp,
    ROUND(AVG(diastolic_bp), 2) AS avg_diastolic_bp,
    ROUND(AVG(sleep_hours), 2) AS avg_sleep_hours,
    ROUND(AVG(age), 2) AS avg_age
FROM nhanes
GROUP BY sleep_group, sex_label
ORDER BY sex_label, sleep_group;
"""
}

for filename, query in queries.items():
    with open(f'../sql/{filename}', 'w') as f:
        f.write(query.strip())
    print(f"Saved: {filename}")

print("\nAll queries exported to sql/ folder!")

Saved: 01_sleep_distribution.sql
Saved: 02_avg_bp_by_sleep.sql
Saved: 03_hypertension_by_age.sql
Saved: 04_summary_by_sex.sql
Saved: 05_summary_by_hypertension.sql
Saved: 06_bp_by_sleep_and_sex.sql

All queries exported to sql/ folder!
